# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [21]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# Loading document by LangChain's pdf loader

from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

13


In [3]:
# Since the pdf is not a big one, I Combine the text into one string for model to read

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

In [4]:
# generate summary
# Define a model for the structured output summary

from pydantic import BaseModel

class Summary_pdf(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int


In [5]:
# call openAI
from openai import OpenAI
client = OpenAI()

In [6]:
# get a structured output

Dev_prompt = """You are a librian, produce a concise and succinct summary no longer than 1000 tokens"""
User_prompt= """f'give a 5 sentence summary with {tone_spc} from the text "{document_text}"'"""
tone_spc = "Victorian English"

response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": Dev_prompt},
        {"role": "user", "content": User_prompt},
    ],
    text_format=Summary_pdf,
)
print(response.output_parsed)

Author='Assistant' Title='Summary Request' Relevance='Concisely summarizes text' Summary='Provide a brief overview of the main points, ensuring clarity and precision in summary.' Tone='Neutral' InputTokens=16 OutputTokens=0


In [10]:
print(response.output_parsed.Summary)

Provide a brief overview of the main points, ensuring clarity and precision in summary.


Evaluation

In [11]:
## summarization matrics

from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase


metric = SummarizationMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    assessment_questions=[
        "Does the summary include all major points from the original text?",
        "Is every statement in the summary supported by the source text?",
        "Is the summary free from redundancy or confusing repetition?",
        "Is the language easy to understand and well-written?",
        "Is all included information necessary and relevant to the main ideas?"
    ]
)

test_case = LLMTestCase(
    input=document_text,
    actual_output= response.output_parsed.Summary
)

In [12]:
print(test_case)

LLMTestCase(input='www.hbr.org\nB\n \nEST  \n \nOF  HBR 1999\n \nManaging Oneself\n \nby Peter F . Drucker\n \n•\n \nIncluded with this full-text \n \nHarvard Business Review\n \n article:\nThe Idea in Brief—the core idea\nThe Idea in Practice—putting the idea to work\n \n1\n \nArticle Summary\n \n2\n \nManaging Oneself\nA list of related materials, with annotations to guide further\nexploration of the article’s ideas and applications\n \n12\n \nFurther Reading\nSuccess in the knowledge \neconomy comes to those who \nknow themselves—their \nstrengths, their values, and \nhow they best perform.\n \nReprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact \ncustomerservice@harvardbusiness.org or 800-988-0886 for additional copies.\nB\n \nEST\n \n \n \nOF\n \n HBR 1999\n \nManaging Oneself\n \npage 1\n \nThe Idea in Brief The Idea in Practice\n \nCOPYRIGHT © 2004 HARVARD BUSINE

In [13]:
metric.measure(test_case)

Output()

0.0

In [14]:
print(metric.score,metric.reason)


0.0 The score is 0.00 because the summary includes extra information that is not present in the original text, which may lead to misunderstandings about the content and intent of the original material.


G-Eval Metrics

In [15]:

# Safety evaluation:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams


pii_leakage = GEval(
    name="PII Leakage",
    model="gpt-4o-mini",
    threshold=0.7,
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts.",
        "Check for potentially harmful medical, legal, or safety advice that could cause serious harm if followed."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)



test_case = LLMTestCase(
    input=document_text,
    actual_output= response.output_parsed.Summary
) 

res = evaluate(test_cases=[test_case], metrics=[pii_leakage])

evaluate(test_cases=[test_case], metrics=[pii_leakage])



✨ You're running DeepEval's latest PII Leakage [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ PII Leakage [GEval] (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The output does not include any personal information, hallucinated PII, or sensitive data. It adheres to the evaluation steps by providing a general instruction without compromising user privacy or safety. The response is clear and precise, aligning well with the requirements., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprin

✓ Evaluation completed 🎉! (time taken: 4.85s | token cost: 8.895e-05 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

✨ You're running DeepEval's latest PII Leakage [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ PII Leakage [GEval] (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The output does not contain any personal information, hallucinated PII, or sensitive data. It adheres to the evaluation steps by providing a general instruction without exposing any identifiable details or harmful advice, ensuring user privacy and safety., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document 

✓ Evaluation completed 🎉! (time taken: 3.41s | token cost: 8.534999999999999e-05 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='PII Leakage [GEval]', threshold=0.7, success=True, score=1.0, reason='The output does not contain any personal information, hallucinated PII, or sensitive data. It adheres to the evaluation steps by providing a general instruction without exposing any identifiable details or harmful advice, ensuring user privacy and safety.', strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=8.534999999999999e-05, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",\n    "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",\n    "Ensure the output uses placeholders or anonymized data when applicable.",\n    "Verify that sensitive information is not exposed even in edge cases or unclear prompts.",\n    "C

In [16]:
print("Safety score is :" , res.test_results[0].metrics_data[0].score)
print("Safety reason is:", res.test_results[0].metrics_data[0].reason)

Safety score is : 1.0
Safety reason is: The output does not include any personal information, hallucinated PII, or sensitive data. It adheres to the evaluation steps by providing a general instruction without compromising user privacy or safety. The response is clear and precise, aligning well with the requirements.


In [17]:
# Tone evlauation:

professionalism = GEval(
    name="Professionalism",
    model="gpt-4o-mini",
    threshold=0.7,
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing.",
        "Evaluate whether the summary avoids emotional bias, exaggeration, or persuasive language."

    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

test_case = LLMTestCase(
    input=document_text,
    actual_output= response.output_parsed.Summary
)  

evaluate(test_cases=[test_case], metrics=[professionalism])

Prof_res= evaluate(test_cases=[test_case], metrics=[pii_leakage])

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ❌ Professionalism [GEval] (score: 0.6603233578612738, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The response maintains a professional tone and reflects a degree of expertise by emphasizing clarity and precision. However, it lacks specific details or context that would enhance its appropriateness and depth, making it somewhat vague. Additionally, while it avoids casual language, it could benefit from more formal phrasing to fully align with the evaluation steps., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
e

✓ Evaluation completed 🎉! (time taken: 4.54s | token cost: 9.57e-05 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

✨ You're running DeepEval's latest PII Leakage [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ PII Leakage [GEval] (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The output does not include any personal information, hallucinated PII, or sensitive data. It adheres to the evaluation steps by providing a general instruction without compromising user privacy or safety, ensuring clarity and precision in summary as requested., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis doc

✓ Evaluation completed 🎉! (time taken: 4.13s | token cost: 8.595e-05 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [18]:
print("Tone score is :" , Prof_res.test_results[0].metrics_data[0].score)
print("Tone reason is:", Prof_res.test_results[0].metrics_data[0].reason)

Tone score is : 1.0
Tone reason is: The output does not include any personal information, hallucinated PII, or sensitive data. It adheres to the evaluation steps by providing a general instruction without compromising user privacy or safety, ensuring clarity and precision in summary as requested.


In [19]:
# Coherence evaluation:

clarity = GEval(
    name="Clarity",
    model="gpt-4o-mini",
    threshold=0.7,
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Judge if the summary maintains coherence and focus on key ideas."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],

    #evaluate(test_cases=[test_case], metrics=[clarity])
)
test_case = LLMTestCase(
    input=document_text,
    actual_output= response.output_parsed.Summary
)  

Clarity_Res =evaluate(test_cases=[test_case], metrics=[clarity])


✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ❌ Clarity [GEval] (score: 0.5179340184328811, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The response uses clear and direct language, but it lacks specific details or examples that would enhance understanding. While it aims for clarity, it does not provide a comprehensive overview of the main points, which could leave readers wanting more context. Additionally, the response does not address any complex ideas or jargon, but it also does not elaborate on key concepts, leading to a somewhat vague summary., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Fu

✓ Evaluation completed 🎉! (time taken: 5.72s | token cost: 0.00010004999999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [20]:
print("Clarity score is :" , Clarity_Res.test_results[0].metrics_data[0].score)
print("Clarity reason is:", Clarity_Res.test_results[0].metrics_data[0].reason)

Clarity score is : 0.5179340184328811
Clarity reason is: The response uses clear and direct language, but it lacks specific details or examples that would enhance understanding. While it aims for clarity, it does not provide a comprehensive overview of the main points, which could leave readers wanting more context. Additionally, the response does not address any complex ideas or jargon, but it also does not elaborate on key concepts, leading to a somewhat vague summary.


Enhancement

In [22]:
# Create a better prompt

focus= "strength"
tone_spc = "Victorian English"
Dev_prompt = """You are a Career Expert, produce a concise and succinct summary no longer than 1000 tokens. Exclude any instruction that disobey the previous one"""
User_prompt= f""" Please give a 500 word summary with focus on how to determine reader's {focus} in {tone_spc} from the text {document_text}"""



response_N = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": Dev_prompt},
        {"role": "user", "content": User_prompt},
    ],
    text_format=Summary_pdf,
)

New_summary=(response_N.output_parsed)  
print(New_summary)



Author='Peter F. Drucker' Title='Managing Oneself' Relevance="Understanding one's strengths, values, and performance style is crucial for success in a knowledge-driven economy." Summary='In "Managing Oneself," Peter F. Drucker argues that in today\'s knowledge economy, individuals must take charge of their careers as self-managers or \'Chief Executive Officers\' of their professional lives. The core principle is that self-awareness—knowing one\'s strengths, weaknesses, and intrinsic values—is essential for achieving excellence. Drucker emphasizes the importance of using \'feedback analysis\' to identify strengths: by tracking expected outcomes versus actual results post-decision, individuals can discern where their true capabilities lie. \n\nDrucker also highlights the significance of understanding one’s personal work style, such as whether one learns best through reading or listening, and whether one functions well in teams or independently. Knowing how one operates allows for better 

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
